Installing

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools
!pip install imbalanced-learn


Libraries

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import os

Train

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize


# === CONFIGURATION ===
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

IMAGE_SIZE = (100, 300)
PATCH_SIZE = 10
stride = 10
NUM_SUBJECTS = 123
NUM_FINGERS = 4
NUM_IMAGES = 5
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]
NUM_ROW_COMPONENTS = 137
NUM_COL_COMPONENTS = 137

# === RIU2 MAPPING ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        table[i] = sum(min_rotation) if transitions <= 2 else P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# === LBP HISTOGRAM ===
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === FEATURE EXTRACTION ===
lbp_images = []
lbp_labels = []


for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Subjects"):
    for img_idx in range(1, NUM_IMAGES + 1):
        for session_path in [base_path_sess1, base_path_sess2]:
            session_name = "1st" if "1st" in session_path else "2nd"
            fused_matrix = []
            complete = True

            for finger_id in range(1, NUM_FINGERS + 1):
                folder = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(session_path, folder, f"{img_idx:02d}.jpg")  # Change to .bmp if needed

                print(f"\n📁 [Subject {subject_id:03d} - Finger {finger_id} - Img {img_idx:02d} - {session_name} Session]")
                print(f"📷 Loading image: {img_path}")

                if not os.path.exists(img_path):
                    print(f"❌ Image does not exist: {img_path}")
                    complete = False
                    break

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Failed to load: {img_path}")
                    complete = False
                    break

                img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img)

                h_patches = (img.shape[0] - PATCH_SIZE) // stride + 1
                w_patches = (img.shape[1] - PATCH_SIZE) // stride + 1
                feature_width = w_patches * sum(P + 2 for _, P in LBP_CONFIGS)

                print(f"🧮 Patches: {h_patches}x{w_patches}, Feature width: {feature_width}")
                lbp_matrix = np.zeros((h_patches, feature_width))

                for i, y in enumerate(range(0, img.shape[0] - PATCH_SIZE + 1, stride)):
                    row_features = []
                    for x in range(0, img.shape[1] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        if block.shape != (PATCH_SIZE, PATCH_SIZE):
                            continue
                        block_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            block_hist.extend(hist)
                        row_features.append(block_hist)
                    if row_features:
                        lbp_matrix[i, :] = np.hstack(row_features)

                fused_matrix.append(lbp_matrix)

            if complete and fused_matrix:
                full_fused = np.hstack(fused_matrix)
                lbp_images.append(full_fused)
                label = f"{subject_id:03d}_img{img_idx:02d}_fused_{session_name}"
                lbp_labels.append(label)
                print(f"✅ Appended fused LBP image with shape: {full_fused.shape}")

# === (2D)^2PCA TRAINING ===
def compute_2d2pca_projection(images, num_row_components, num_col_components):
    print("\n⚙️ Computing (2D)^2PCA projection matrices...")
    n = len(images)
    h, w = images[0].shape
    print(f"📏 Matrix shape: {h}x{w}, Num samples: {n}")
    mean_img = sum(images) / n
    G_row = np.zeros((h, h))
    G_col = np.zeros((w, w))
    for A in images:
        A = A - mean_img
        G_row += A @ A.T
        G_col += A.T @ A
    G_row /= n
    G_col /= n
    eig_vals_r, eig_vecs_r = np.linalg.eigh(G_row)
    eig_vals_c, eig_vecs_c = np.linalg.eigh(G_col)
    U = eig_vecs_r[:, np.argsort(-eig_vals_r)[:num_row_components]]
    V = eig_vecs_c[:, np.argsort(-eig_vals_c)[:num_col_components]]
    print(f"✅ U shape: {U.shape}, V shape: {V.shape}")
    return U, V

if len(lbp_images) == 0:
    print("❌ No images were processed. Check image paths and extensions.")
else:
    U, V = compute_2d2pca_projection(lbp_images, NUM_ROW_COMPONENTS, NUM_COL_COMPONENTS)

    # === PROJECT IMAGES ===
    print("\n📤 Projecting all samples into (2D)^2PCA space...")
    projected_features = [U.T @ A @ V for A in lbp_images]
    flat_features = np.array([f.flatten() for f in projected_features])
    flat_features = normalize(flat_features, norm='l2')
    lbp_labels = np.array(lbp_labels)

    print("\n✅ Final (2D)^2PCA-transformed shape:", flat_features.shape)
    print("📌 Sample labels:", lbp_labels[:5])


Test

In [ ]:
# Modified test extraction and projection code for FV-USM P1-S1 using (2D)^2PCA with tracking prints

import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern

# === CONFIGURATION ===
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

IMAGE_SIZE = (100, 300)
PATCH_SIZE = 10
stride = 10
TEST_INDICES = [6]
NUM_FINGERS = 4
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]

# === RIU2 MAPPING ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        table[i] = sum(min_rotation) if transitions <= 2 else P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# === LBP HISTOGRAM ===
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === EXTRACT TEST SET ===
test_lbp_images = []
test_labels = []

for subject_id in tqdm(range(1, 124), desc="Subjects"):
    for img_idx in TEST_INDICES:
        for session_path in [base_path_sess1, base_path_sess2]:
            session_name = "1st" if "1st" in session_path else "2nd"
            combined_matrix = []
            complete = True

            for finger_id in range(1, NUM_FINGERS + 1):
                folder_name = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(session_path, folder_name, f"{img_idx:02d}.jpg")
                print(f"\n📥 Reading: {img_path}")

                if not os.path.exists(img_path):
                    print(f"❌ File missing: {img_path}")
                    complete = False
                    break

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Cannot read image: {img_path}")
                    complete = False
                    break

                img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img)

                h_patches = (img.shape[0] - PATCH_SIZE) // stride + 1
                w_patches = (img.shape[1] - PATCH_SIZE) // stride + 1

                lbp_matrix = np.zeros((h_patches, w_patches * sum(P + 2 for _, P in LBP_CONFIGS)))

                for i, y in enumerate(range(0, img.shape[0] - PATCH_SIZE + 1, stride)):
                    row_features = []
                    for x in range(0, img.shape[1] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        if block.shape != (PATCH_SIZE, PATCH_SIZE):
                            continue
                        block_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            block_hist.extend(hist)
                        row_features.append(block_hist)
                    if row_features:
                        lbp_matrix[i, :] = np.hstack(row_features)

                combined_matrix.append(lbp_matrix)

            if complete and len(combined_matrix) == NUM_FINGERS:
                fused_lbp = np.hstack(combined_matrix)
                test_lbp_images.append(fused_lbp)
                label = f"{subject_id:03d}_img{img_idx:02d}_fused_{session_name}"
                test_labels.append(label)
                print(f"✅ Extracted features for: {label}")

# === PROJECT TEST SET ===
try:
    test_projected = [U.T @ A @ V for A in test_lbp_images]
    test_flat_features = np.array([f.flatten() for f in test_projected])
    test_labels = np.array(test_labels)

    print("\n✅ Test projection completed.")
    print("📐 Projected feature shape:", test_flat_features.shape)
    print("🧾 Sample test labels:", test_labels[:5])
except NameError:
    print("❌ U and V projection matrices not found. Run training code first.")


Benchmarking1

In [ ]:
# === SESSION-SENSITIVE EVALUATION ===
correct_matches = 0
total_tests = len(test_labels)

print("\n🔍 Starting Session-Sensitive Classification using Manhattan distance...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_label = test_labels[i]  # e.g., "123_img03_fused_2nd"

    # 📏 Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    closest_index = np.argmin(distances)
    predicted_label = lbp_labels[closest_index]  # e.g., "123_img01_fused_1st"

    # 🎯 Extract subject ID and session
    true_parts = true_label.split('_')       # ['123', 'img03', 'fused', '2nd']
    pred_parts = predicted_label.split('_')  # ['123', 'img01', 'fused', '1st']

    true_id = true_parts[0]
    true_session = true_parts[-1]
    pred_id = pred_parts[0]
    pred_session = pred_parts[-1]

    if (true_id == pred_id) and (true_session == pred_session):
        correct_matches += 1
        result = "✅ CORRECT"
        emoji = "🎯"
    else:
        result = "❌ WRONG"
        emoji = "⚠️"

    print(f"\n{emoji} Sample {i+1:03d} / {total_tests}")
    print(f"    🧾 Predicted: {predicted_label}")
    print(f"    🎯 Actual   : {true_label}")
    print(f"    📌 Match    : {result}")

# === FINAL ACCURACY ===
accuracy = (correct_matches / total_tests) * 100 if total_tests > 0 else 0
print("\n📊 Session-Sensitive Classification Report")
print(f"✅ Correct matches: {correct_matches} / {total_tests}")
print(f"🎯 Accuracy       : {accuracy:.2f}%")


Benchmarking2

In [ ]:
# === SESSION-INDEPENDENT EVALUATION ===
correct_matches = 0
total_tests = len(test_labels)

print("\n🔍 Starting Session-Independent Classification using Manhattan distance...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_label = test_labels[i]  # e.g., "007_img03_fused_2nd"

    # 📏 Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    closest_index = np.argmin(distances)
    predicted_label = lbp_labels[closest_index]  # e.g., "007_img01_fused_1st"

    # 🎯 Compare subject ID only
    true_id = true_label.split('_')[0]
    pred_id = predicted_label.split('_')[0]

    if pred_id == true_id:
        correct_matches += 1
        result = "✅ MATCH"
        emoji = "🎯"
    else:
        result = "❌ MISMATCH"
        emoji = "⚠️"

    print(f"\n{emoji} Test sample {i+1:03d} / {total_tests}")
    print(f"    🔑 Predicted: {predicted_label}")
    print(f"    🎯 Actual   : {true_label}")
    print(f"    ➡️  Match    : {result}")

# === FINAL REPORT ===
accuracy = (correct_matches / total_tests) * 100 if total_tests > 0 else 0
print("\n📊 Session-Independent Classification Report")
print(f"✅ Correct Matches : {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy : {accuracy:.2f}%")


Session Sensitive R5

In [ ]:
import numpy as np
from collections import defaultdict

ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating Session-Sensitive CMC (Rank-1 & Rank-5)...")

for i in range(total_tests):
    test_label = test_labels[i]
    proj_test = test_flat_features[i]

    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_session = test_parts[-1]
    test_id = f"{test_subject}_{test_session}"

    # 📏 Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    matched = False
    for r in range(1, max(ranks) + 1):
        candidate_label = lbp_labels[sorted_indices[r - 1]]
        parts = candidate_label.split('_')
        candidate_subject = parts[0]
        candidate_session = parts[-1]
        candidate_id = f"{candidate_subject}_{candidate_session}"

        if candidate_id == test_id and not matched:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            matched = True

# === FINAL RESULTS ===
for k in ranks:
    accuracy = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy (Subject + Session): {accuracy:.2f}%")


Session Sensitive CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_labels)

print("📊 Calculating Session-Sensitive CMC Curve (Rank-1 to Rank-100)...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    test_parts = test_labels[i].split('_')
    true_subject = test_parts[0]
    true_session = test_parts[-1]
    true_id = f"{true_subject}_{true_session}"

    # Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    for r in range(max_rank):
        candidate_label = lbp_labels[sorted_indices[r]]
        cand_parts = candidate_label.split('_')
        cand_subject = cand_parts[0]
        cand_session = cand_parts[-1]
        candidate_id = f"{cand_subject}_{cand_session}"

        if candidate_id == true_id:
            rank_correct[r:] += 1
            break

# === Normalize
cmc_curve = (rank_correct / total_tests) * 100

# === Plot
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Sensitive CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Sensitive CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) + (2D)$^2$PCA (Strategy 2, Protocol 1)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank+1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# === Print key ranks
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Sensitive Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter

# === Toggle SMOTE ===
use_smote = True  # Set True if you want to apply SMOTE

print("\n📊 Evaluating (Session-Sensitive) LBP((8,1),(16,1),(8,2)) + (2D)^2PCA...\n")

all_scores = []
all_labels = []

# === Loop through test and train samples
for i in range(len(test_flat_features)):
    test_vec = test_flat_features[i]
    test_parts = test_labels[i].split('_')
    test_subject = test_parts[0]
    test_session = test_parts[-1]

    for j in range(len(flat_features)):
        train_vec = flat_features[j]
        train_parts = lbp_labels[j].split('_')
        train_subject = train_parts[0]
        train_session = train_parts[-1]

        # === Compute negative Manhattan distance (higher = more similar)
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # === Label as genuine if both subject and session match
        is_genuine = int(test_subject == train_subject and test_session == train_session)
        all_labels.append(is_genuine)

# === Convert to NumPy arrays
scores = np.array(all_scores).reshape(-1, 1)
labels = np.array(all_labels)

print("🔢 Original label distribution:", Counter(labels))

# === Normalize scores to [0, 1]
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Apply SMOTE (optional)
if use_smote:
    smote = SMOTE(random_state=42)
    scores, labels = smote.fit_resample(scores, labels)
    print("🧪 After SMOTE label distribution:", Counter(labels))

# === Threshold Sweeping to find best F1
best_f1 = best_thresh = best_prec = best_rec = 0
for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = prec
        best_rec = rec

# === Final evaluation
final_preds = (scores >= best_thresh).astype(int)
acc = accuracy_score(labels, final_preds)

# === Print results
print("\n🔍 Summary (Session-Sensitive)")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {acc * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")


Session Independent R5

In [ ]:
from collections import defaultdict
import numpy as np

ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating CMC, Rank-1, and Rank-5 (Session-Independent)...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_id = test_labels[i].split('_')[0]  # 🔍 Subject only

    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    top_k_indices = np.argsort(distances)

    found = False
    for r in range(1, max(ranks) + 1):
        candidate_label = lbp_labels[top_k_indices[r - 1]]
        candidate_id = candidate_label.split('_')[0]  # 🔍 Subject only

        if candidate_id == true_id and not found:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            found = True

# === Final Results ===
for k in ranks:
    cmc_score = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy: {cmc_score:.2f}%")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
correct_matches = 0
total_tests = len(test_labels)

print("📊 Calculating Session-Independent CMC Curve (Rank-1 to Rank-100)...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_subject = test_labels[i].split('_')[0]  # Subject ID only

    # Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # === Rank-1 Classification (with subject matching)
    for idx in sorted_indices:
        pred_subject = lbp_labels[idx].split('_')[0]
        if pred_subject == true_subject:
            if idx == sorted_indices[0]:
                correct_matches += 1  # ✅ Only if correct match is at Rank-1
            break  # Stop after first match for classification

    # === CMC Curve (match subject at correct rank)
    for r in range(max_rank):
        candidate_subject = lbp_labels[sorted_indices[r]].split('_')[0]
        if candidate_subject == true_subject:
            rank_correct[r:] += 1
            break

# === Normalize CMC to percentage
cmc_curve = (rank_correct / total_tests) * 100
rank1_accuracy = (correct_matches / total_tests) * 100

# === Plotting
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Independent CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Independent CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) + (2D)$^2$PCA (Strategy 2, Protocol 1)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# === Final Report
print("\n📊 Session-Independent Classification Report")
print(f"✅ Rank-1 Classification Accuracy : {rank1_accuracy:.2f}%")
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter

# === Toggle SMOTE ===
use_smote = True  # Set True if you want to apply SMOTE

print("\n📊 Evaluating (Session-Independent) LBP((8,1),(16,1),(8,2)) + (2D)^2PCA...\n")

all_scores = []
all_labels = []

# === Loop through test and train samples
for i in range(len(test_flat_features)):
    test_vec = test_flat_features[i]
    test_parts = test_labels[i].split('_')
    test_subject = test_parts[0]  # Only subject ID

    for j in range(len(flat_features)):
        train_vec = flat_features[j]
        train_parts = lbp_labels[j].split('_')
        train_subject = train_parts[0]  # Only subject ID

        # === Compute negative Manhattan distance (higher = more similar)
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # === Label as genuine if subject IDs match (ignore session)
        is_genuine = int(test_subject == train_subject)
        all_labels.append(is_genuine)

# === Convert to NumPy arrays
scores = np.array(all_scores).reshape(-1, 1)
labels = np.array(all_labels)

print("🔢 Original label distribution:", Counter(labels))

# === Normalize scores to [0, 1]
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Apply SMOTE (optional)
if use_smote:
    smote = SMOTE(random_state=42)
    scores, labels = smote.fit_resample(scores, labels)
    print("🧪 After SMOTE label distribution:", Counter(labels))

# === Threshold Sweeping to find best F1
best_f1 = best_thresh = best_prec = best_rec = 0
for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = prec
        best_rec = rec

# === Final evaluation
final_preds = (scores >= best_thresh).astype(int)
acc = accuracy_score(labels, final_preds)

# === Print results
print("\n🔍 Summary (Session-Independent)")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {acc * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
